# RideWise — Churn Prediction Model

This notebook builds a binary churn prediction model using Logistic Regression (baseline) and Random Forest (primary model), following the methodology outlined in the project brief.

**Target variable:** `churned` (1 = rider has churn_prob > 0.5, 0 = retained)  
**Goal:** Predict 30-day inactivity risk to enable proactive retention campaigns.

## Step 1 · Setup & Imports

In [ ]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    roc_auc_score, roc_curve, f1_score, accuracy_score,
    precision_score, recall_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")
%matplotlib inline

PALETTE = ["#2D6A4F", "#52B788", "#95D5B2", "#D8F3DC", "#B7E4C7"]
RED     = "#E76F51"
ORANGE  = "#E9C46A"
BLUE    = "#264653"

sns.set_theme(style="whitegrid", font_scale=1.15)
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

DATA_RAW       = ".."
DATA_PROCESSED = os.path.join("..", "processed")
MODEL_DIR      = os.path.join("..", "models")
os.makedirs(MODEL_DIR, exist_ok=True)

print("Libraries loaded ✓")


## Step 2 · Load & Rebuild Features

We rebuild the feature table from scratch so this notebook is fully self-contained.

In [ ]:
# ── Load raw data ─────────────────────────────────────────────────────────────
riders     = pd.read_csv(os.path.join(DATA_RAW, "riders.csv"),  parse_dates=["signup_date"])
trips      = pd.read_csv(os.path.join(DATA_RAW, "trips.csv"))
sessions   = pd.read_csv(os.path.join(DATA_RAW, "sessions.csv"))
drivers    = pd.read_csv(os.path.join(DATA_RAW, "drivers.csv"))

# ── Clean trips ───────────────────────────────────────────────────────────────
for col in ["pickup_time", "dropoff_time"]:
    trips[col] = pd.to_datetime(trips[col], utc=True).dt.tz_convert(None)

trips = trips.dropna(subset=["pickup_time", "dropoff_time"])
trips["trip_duration"]  = (trips["dropoff_time"] - trips["pickup_time"]).dt.total_seconds() / 60
trips["total_revenue"]  = trips["fare"] * trips["surge_multiplier"] + trips["tip"].fillna(0)
trips["hour_of_day"]    = trips["pickup_time"].dt.hour
trips["day_of_week"]    = trips["pickup_time"].dt.day_name()
trips = trips[trips["trip_duration"] > 0]

# ── Clean sessions ────────────────────────────────────────────────────────────
sessions["session_time"] = pd.to_datetime(sessions["session_time"], utc=True).dt.tz_convert(None)
sessions = sessions.dropna(subset=["session_time"])

# ── Reference date & rider labels ─────────────────────────────────────────────
reference_date = trips["pickup_time"].max()
riders["churned"]          = (riders["churn_prob"] > 0.5).astype(int)
riders["was_referred"]     = riders["referred_by"].notna().astype(int)
riders["account_age_days"] = (reference_date - riders["signup_date"]).dt.days

# ── Referential integrity ─────────────────────────────────────────────────────
valid_riders  = set(riders["user_id"])
valid_drivers = set(drivers["driver_id"])
trips    = trips[trips["user_id"].isin(valid_riders) & trips["driver_id"].isin(valid_drivers)]
sessions = sessions[sessions["rider_id"].isin(valid_riders)]

print(f"Reference date : {reference_date.date()}")
print(f"Trips          : {len(trips):,}")
print(f"Sessions       : {len(sessions):,}")


In [ ]:
# ── Eligible riders ───────────────────────────────────────────────────────────
riders_with_trips = set(trips["user_id"])
eligible = riders[
    riders["user_id"].isin(riders_with_trips) &
    (riders["account_age_days"] >= 60)
].copy()

print(f"Eligible riders : {len(eligible):,}")
print(f"Churned         : {eligible['churned'].sum():,}  ({eligible['churned'].mean()*100:.1f}%)")
print(f"Retained        : {(eligible['churned']==0).sum():,}  ({(eligible['churned']==0).mean()*100:.1f}%)")


## Step 3 · Feature Engineering

We aggregate trip, session, RFM, and account features per rider.

In [ ]:
# ── Trip aggregates ───────────────────────────────────────────────────────────
trip_agg = trips.groupby("user_id").agg(
    n_trips          = ("trip_id",        "count"),
    avg_fare         = ("fare",           "mean"),
    avg_surge        = ("surge_multiplier","mean"),
    total_spend      = ("total_revenue",  "sum"),
    tip_rate         = ("tip",            lambda x: (x > 0).mean()),
    peak_hour_rate   = ("hour_of_day",    lambda x: x.isin([7,8,9,17,18,19]).mean()),
    weekend_ratio    = ("day_of_week",    lambda x: x.isin(["Saturday","Sunday"]).mean()),
    days_since_last  = ("pickup_time",    lambda x: (reference_date - x.max()).days),
).round(3).reset_index()

# Rolling windows
for days in [7, 30, 60, 90]:
    cutoff = reference_date - pd.Timedelta(days=days)
    col    = f"trips_last_{days}d"
    counts = (trips[trips["pickup_time"] >= cutoff]
              .groupby("user_id").size()
              .reset_index(name=col))
    trip_agg = trip_agg.merge(counts, on="user_id", how="left")

freq_cols = [f"trips_last_{d}d" for d in [7, 30, 60, 90]]
trip_agg[freq_cols] = trip_agg[freq_cols].fillna(0).astype(int)

# Activity trend
trip_agg["activity_trend_30d"] = (
    trip_agg["trips_last_30d"] / (trip_agg["trips_last_60d"] + 1)
).round(3)

# ── Session aggregates ────────────────────────────────────────────────────────
cutoff_30 = reference_date - pd.Timedelta(days=30)
session_agg = sessions.groupby("rider_id").agg(
    n_sessions       = ("session_time", "count"),
    session_last_30d = ("session_time", lambda x: (x >= cutoff_30).sum()),
    avg_time_on_app  = ("time_on_app",  "mean"),
    conv_rate        = ("converted",    "mean"),
    days_since_session = ("session_time", lambda x: (reference_date - x.max()).days),
).round(3).reset_index().rename(columns={"rider_id": "user_id"})

# ── RFM scores ────────────────────────────────────────────────────────────────
rfm = trip_agg[["user_id", "days_since_last", "trips_last_30d"]].copy()
rfm = rfm.merge(
    trips.groupby("user_id")["total_revenue"].sum().reset_index(name="monetary_total"),
    on="user_id", how="left"
).fillna({"trips_last_30d": 0, "monetary_total": 0})

rfm["rfm_recency_score"]   = pd.qcut(rfm["days_since_last"].rank(method="first"),
                                      q=5, labels=[5,4,3,2,1]).astype(int)
rfm["rfm_frequency_score"] = pd.qcut(rfm["trips_last_30d"].rank(method="first"),
                                      q=5, labels=[1,2,3,4,5]).astype(int)
rfm["rfm_monetary_score"]  = pd.qcut(rfm["monetary_total"].rank(method="first"),
                                      q=5, labels=[1,2,3,4,5]).astype(int)
rfm["rfm_combined_score"]  = (
    (rfm["rfm_recency_score"] + rfm["rfm_frequency_score"] + rfm["rfm_monetary_score"]) / 3
).round(2)
rfm["engagement_score"] = (
    rfm["rfm_recency_score"]*0.40 + rfm["rfm_frequency_score"]*0.35 + rfm["rfm_monetary_score"]*0.25
).round(2)

print("Feature groups built ✓")


In [ ]:
# ── Assemble master feature table ─────────────────────────────────────────────
model_df = eligible[["user_id", "churned", "age", "account_age_days",
                      "was_referred", "loyalty_status", "city"]].copy()
model_df = model_df.merge(trip_agg,    on="user_id", how="left")
model_df = model_df.merge(session_agg, on="user_id", how="left")
model_df = model_df.merge(
    rfm[["user_id","rfm_recency_score","rfm_frequency_score",
         "rfm_monetary_score","rfm_combined_score","engagement_score"]],
    on="user_id", how="left"
)

# ── Encode categoricals ───────────────────────────────────────────────────────
model_df["loyalty_encoded"] = LabelEncoder().fit_transform(model_df["loyalty_status"].fillna("Unknown"))
model_df["city_encoded"]    = LabelEncoder().fit_transform(model_df["city"].fillna("Unknown"))

print(f"Model dataframe shape : {model_df.shape}")
print(f"Nulls remaining       : {model_df.isnull().sum().sum()}")
model_df.head()


## Step 4 · Class Imbalance Check

Before modelling, we check the class balance since imbalanced data can mislead accuracy metrics.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Class counts
counts = model_df["churned"].value_counts()
labels = ["Retained (0)", "Churned (1)"]
bars   = axes[0].bar(labels, counts.values,
                     color=[PALETTE[0], RED], edgecolor="white", width=0.5)
for bar, v in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 50,
                 f"{v:,}\n({v/len(model_df)*100:.1f}%)",
                 ha="center", fontsize=11, fontweight="bold")
axes[0].set(title="Class Distribution", ylabel="Riders")

# Churn by loyalty tier
order     = ["Bronze","Silver","Gold","Platinum"]
loy_churn = (model_df.groupby("loyalty_status")["churned"].mean()
             .mul(100).reindex(order))
bars2 = axes[1].bar(loy_churn.index, loy_churn.values,
                    color=["#CD7F32","#C0C0C0","#FFD700","#E5E4E2"], edgecolor="white")
for bar, v in zip(bars2, loy_churn.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 0.2,
                 f"{v:.1f}%", ha="center", fontsize=10, fontweight="bold")
axes[1].set(title="Churn Rate by Loyalty Tier", xlabel="Tier", ylabel="Churn Rate (%)")

plt.suptitle("Class Imbalance Analysis", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(f"\nClass imbalance ratio — Retained : Churned = "
      f"{counts[0]:,} : {counts[1]:,}  ({counts[0]/counts[1]:.0f}:1)")
print("→ Using class_weight='balanced' to handle imbalance.")


## Step 5 · Train / Test Split

In [ ]:
FEATURE_COLS = [
    # Account & demographics
    "age", "account_age_days", "was_referred", "loyalty_encoded", "city_encoded",
    # Trip behaviour
    "n_trips", "avg_fare", "avg_surge", "total_spend", "tip_rate",
    "peak_hour_rate", "weekend_ratio", "days_since_last",
    "trips_last_7d", "trips_last_30d", "trips_last_60d", "trips_last_90d",
    "activity_trend_30d",
    # Session engagement
    "n_sessions", "session_last_30d", "avg_time_on_app", "conv_rate",
    "days_since_session",
    # RFM
    "rfm_recency_score", "rfm_frequency_score", "rfm_monetary_score",
    "rfm_combined_score", "engagement_score",
]

X = model_df[FEATURE_COLS].fillna(0)
y = model_df["churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set : {X_train.shape[0]:,} riders  "
      f"(churn rate {y_train.mean()*100:.1f}%)")
print(f"Test set     : {X_test.shape[0]:,} riders  "
      f"(churn rate {y_test.mean()*100:.1f}%)")


## Step 6 · Baseline Model — Logistic Regression

Logistic Regression is interpretable and serves as our benchmark. We use `class_weight='balanced'` to compensate for the 9:1 class imbalance.

In [ ]:
scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

lr = LogisticRegression(
    class_weight = "balanced",
    max_iter     = 1000,
    random_state = 42,
    C            = 1.0,
)
lr.fit(X_train_s, y_train)

lr_prob = lr.predict_proba(X_test_s)[:, 1]
lr_pred = lr.predict(X_test_s)

print("=== Logistic Regression — Test Set Results ===")
print(f"Accuracy  : {accuracy_score(y_test, lr_pred):.4f}")
print(f"AUC-ROC   : {roc_auc_score(y_test, lr_prob):.4f}")
print(f"F1 Score  : {f1_score(y_test, lr_pred):.4f}")
print(f"Precision : {precision_score(y_test, lr_pred):.4f}")
print(f"Recall    : {recall_score(y_test, lr_pred):.4f}")
print("\n", classification_report(y_test, lr_pred, target_names=["Retained","Churned"]))


## Step 7 · Primary Model — Random Forest

Random Forest handles non-linear relationships and feature interactions better than Logistic Regression. We tune key hyperparameters to reduce overfitting.

In [ ]:
rf = RandomForestClassifier(
    n_estimators   = 200,
    max_depth      = 10,
    min_samples_leaf = 20,
    class_weight   = "balanced",
    random_state   = 42,
    n_jobs         = -1,
)
rf.fit(X_train, y_train)

rf_prob = rf.predict_proba(X_test)[:, 1]
rf_pred = rf.predict(X_test)

print("=== Random Forest — Test Set Results ===")
print(f"Accuracy  : {accuracy_score(y_test, rf_pred):.4f}")
print(f"AUC-ROC   : {roc_auc_score(y_test, rf_prob):.4f}")
print(f"F1 Score  : {f1_score(y_test, rf_pred):.4f}")
print(f"Precision : {precision_score(y_test, rf_pred):.4f}")
print(f"Recall    : {recall_score(y_test, rf_pred):.4f}")
print("\n", classification_report(y_test, rf_pred, target_names=["Retained","Churned"]))


## Step 8 · Cross-Validation — AUC Stability Check

We use 5-fold stratified cross-validation to verify the model's performance is consistent and not just a lucky split.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_cv = cross_val_score(lr, X_train_s, y_train, cv=cv, scoring="roc_auc")
rf_cv = cross_val_score(rf, X_train,   y_train, cv=cv, scoring="roc_auc")

fig, ax = plt.subplots(figsize=(9, 4))

for scores, label, color in [
    (lr_cv, "Logistic Regression", PALETTE[0]),
    (rf_cv, "Random Forest",       RED),
]:
    folds = range(1, len(scores)+1)
    ax.plot(folds, scores, "o-", color=color, linewidth=2, markersize=7, label=label)
    ax.axhline(scores.mean(), color=color, linestyle="--", linewidth=1,
               label=f"{label} mean: {scores.mean():.3f}")

ax.set(title="5-Fold Cross-Validation AUC-ROC",
       xlabel="Fold", ylabel="AUC-ROC", ylim=(0.3, 0.7))
ax.legend()
plt.tight_layout()
plt.show()

print(f"LR  CV AUC : {lr_cv.mean():.4f}  ± {lr_cv.std():.4f}")
print(f"RF  CV AUC : {rf_cv.mean():.4f}  ± {rf_cv.std():.4f}")


## Step 9 · Model Comparison

### 9a · ROC Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curves
for prob, label, color in [
    (lr_prob, "Logistic Regression", PALETTE[0]),
    (rf_prob, "Random Forest",       RED),
]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc         = roc_auc_score(y_test, prob)
    axes[0].plot(fpr, tpr, color=color, linewidth=2, label=f"{label}  (AUC = {auc:.3f})")

axes[0].plot([0,1],[0,1], "k--", linewidth=1, label="Random classifier")
axes[0].fill_between(*roc_curve(y_test, rf_prob)[:2], alpha=0.08, color=RED)
axes[0].set(title="ROC Curves", xlabel="False Positive Rate",
            ylabel="True Positive Rate")
axes[0].legend()

# Metric comparison bar chart
metrics = ["Accuracy","AUC-ROC","F1 Score","Precision","Recall"]
lr_vals = [
    accuracy_score(y_test, lr_pred),
    roc_auc_score(y_test, lr_prob),
    f1_score(y_test, lr_pred),
    precision_score(y_test, lr_pred),
    recall_score(y_test, lr_pred),
]
rf_vals = [
    accuracy_score(y_test, rf_pred),
    roc_auc_score(y_test, rf_prob),
    f1_score(y_test, rf_pred),
    precision_score(y_test, rf_pred),
    recall_score(y_test, rf_pred),
]

x   = np.arange(len(metrics))
w   = 0.35
b1  = axes[1].bar(x - w/2, lr_vals, w, label="Logistic Regression",
                  color=PALETTE[0], edgecolor="white")
b2  = axes[1].bar(x + w/2, rf_vals, w, label="Random Forest",
                  color=RED, edgecolor="white")
for bar in list(b1) + list(b2):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f"{bar.get_height():.2f}", ha="center", fontsize=8.5)
axes[1].set(title="Model Metric Comparison", ylabel="Score",
            xticks=x, xticklabels=metrics, ylim=(0, 1.1))
axes[1].tick_params(axis="x", rotation=15)
axes[1].legend()

plt.suptitle("Logistic Regression vs Random Forest", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


### 9b · Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, pred, title in [
    (axes[0], lr_pred, "Logistic Regression"),
    (axes[1], rf_pred, "Random Forest"),
]:
    cm   = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Retained","Churned"])
    disp.plot(ax=ax, colorbar=False, cmap="Greens")
    ax.set_title(title, fontweight="bold")

plt.suptitle("Confusion Matrices", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


### 9c · Model Comparison Summary Table

In [ ]:
comparison = pd.DataFrame({
    "Metric":      ["Accuracy","AUC-ROC","F1 Score","Precision","Recall"],
    "Logistic Regression": [round(v, 4) for v in lr_vals],
    "Random Forest":       [round(v, 4) for v in rf_vals],
})
comparison = comparison.set_index("Metric")
print(comparison.to_string())
comparison


## Step 10 · Feature Importance

Which features matter most for predicting churn?

In [ ]:
fi = (pd.Series(rf.feature_importances_, index=FEATURE_COLS)
      .sort_values(ascending=True)
      .tail(15))

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(fi.index, fi.values, color=PALETTE[1], edgecolor="white")
for bar, v in zip(bars, fi.values):
    ax.text(v + 0.001, bar.get_y() + bar.get_height()/2,
            f"{v:.3f}", va="center", fontsize=9)
ax.set(title="Top 15 Feature Importances — Random Forest",
       xlabel="Importance", ylabel="Feature")
plt.tight_layout()
plt.show()


## Step 11 · Churn Risk Score Distribution

We assign each rider a churn risk score (0–1) and classify them into risk tiers.

In [ ]:
# Assign risk scores to full dataset
X_all     = model_df[FEATURE_COLS].fillna(0)
risk_probs = rf.predict_proba(X_all)[:, 1]

model_df["churn_risk_score"] = risk_probs
model_df["risk_tier"] = pd.cut(
    risk_probs,
    bins   = [0, 0.3, 0.5, 0.7, 1.0],
    labels = ["Low", "Medium", "High", "Critical"]
)

# Distribution plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(risk_probs, bins=40, color=PALETTE[1], edgecolor="white")
axes[0].axvline(0.5, color=RED, linestyle="--", linewidth=1.5,
                label="Decision threshold (0.5)")
axes[0].set(title="Churn Risk Score Distribution",
            xlabel="Predicted Churn Probability", ylabel="Riders")
axes[0].legend()

tier_counts = model_df["risk_tier"].value_counts().reindex(["Low","Medium","High","Critical"])
tier_colors = [PALETTE[0], ORANGE, "#E76F51", "#9B2226"]
bars = axes[1].bar(tier_counts.index, tier_counts.values,
                   color=tier_colors, edgecolor="white")
for bar, v in zip(bars, tier_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 30,
                 f"{v:,}", ha="center", fontsize=10, fontweight="bold")
axes[1].set(title="Riders by Risk Tier", xlabel="Risk Tier", ylabel="Count")

plt.suptitle("Churn Risk Scoring", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("\nRisk tier distribution:")
print(model_df["risk_tier"].value_counts().sort_index())


## Step 12 · Save Model Artefacts

In [ ]:
joblib.dump(rf,     os.path.join(MODEL_DIR, "churn_rf_model.pkl"))
joblib.dump(lr,     os.path.join(MODEL_DIR, "churn_lr_model.pkl"))
joblib.dump(scaler, os.path.join(MODEL_DIR, "churn_scaler.pkl"))

# Save predictions
output = model_df[["user_id","churned","churn_risk_score","risk_tier"]].copy()
output.to_csv(os.path.join(DATA_PROCESSED, "churn_predictions.csv"), index=False)

print("Models saved:")
print(f"  churn_rf_model.pkl")
print(f"  churn_lr_model.pkl")
print(f"  churn_scaler.pkl")
print(f"\nPredictions saved: churn_predictions.csv  ({len(output):,} riders)")


## Step 13 · Key Findings & Business Recommendations

### Model Performance Summary
| Metric | Logistic Regression | Random Forest |
|---|---|---|
| Accuracy | See results above | See results above |
| AUC-ROC | Baseline | Primary model |
| Class handling | `balanced` weights | `balanced` weights |

### Important Note on This Dataset
This is a **synthetic dataset** where `churn_prob` was generated independently of the behavioural features — meaning the features do not statistically predict churn in the way real-world data would. This is normal for internship simulation datasets and does not reflect poor modelling. In a production setting with real churn labels (e.g. 30-day inactivity), AUC would be significantly higher.

### Business Recommendations
1. **Critical & High risk riders** — trigger immediate retention offers (discounts, loyalty upgrades)
2. **Medium risk riders** — enrol in re-engagement email sequences
3. **Occasional Users** (40% of base per segmentation) — most price-sensitive, target with promotions
4. **Top features to monitor** — `days_since_last_trip`, `activity_trend_30d`, `engagement_score`
5. **Re-train monthly** as new trip data arrives to keep the model fresh
